# San Fernando ITT - Geovisor Interactivo

Notebook para el analisis geoespacial del Corredor San Fernando.

**Flujo de trabajo:**
1. Configuracion del entorno (Colab)
2. Carga del shapefile (MAGNA-SIRGAS) y transformacion a WGS84
3. Validacion de calidad de datos
4. Conversion a GeoJSON
5. Geovisor interactivo con edicion de poligono
6. Captura de geometria editada y descarga
7. Extraccion de POIs de OpenStreetMap
8. Tabla de POIs en el area de estudio
9. Exportacion de resultados

## 1. Configuracion del entorno

In [1]:
import sys
from pathlib import Path

EN_COLAB = 'google.colab' in sys.modules

if EN_COLAB:
    try:
        import mapclassify
    except ImportError:
        %pip install -q mapclassify
    # Siempre clonar fresco para tener la ultima version
    !rm -rf Corredor_San_fernando
    !git clone --depth 1 https://github.com/j0rg3c45/Corredor_San_fernando.git
    ruta_base = Path('Corredor_San_fernando')
else:
    ruta_base = Path('..')

print(f"Entorno: {'Google Colab' if EN_COLAB else 'Local'}")
print(f"Ruta base: {ruta_base.resolve()}")

Cloning into 'Corredor_San_fernando'...
remote: Enumerating objects: 29, done.
remote: Counting objects: 100% (29/29), done.
remote: Compressing objects: 100% (24/24), done.
remote: Total 29 (delta 1), reused 21 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (29/29), 23.38 KiB | 392.00 KiB/s, done.
Resolving deltas: 100% (1/1), done.
Entorno: Google Colab
Ruta base: /content/Corredor_San_fernando


## 2. Importaciones

In [2]:
import geopandas as gpd
import pandas as pd
import folium
from folium.plugins import Draw
import json
import requests

## 3. Configuracion de rutas y carga del shapefile

In [3]:
# Rutas del proyecto
ruta_shapefile = ruta_base / 'Data' / 'shape_geo' / 'Corredor_San_fernando.shp'
ruta_geojson_fuente = ruta_base / 'Data' / 'Geojson_Corredor_San_fernando' / 'geojson_Corredor_San_fernando.geojson'
ruta_geojson = ruta_base / 'Data' / 'Corredor_San_fernando_wgs84.geojson'
ruta_outputs = ruta_base / 'Outputs'
ruta_outputs.mkdir(parents=True, exist_ok=True)

# Priorizar el GeoJSON actualizado; si no existe, usar el shapefile
if ruta_geojson_fuente.exists():
    ruta_entrada = ruta_geojson_fuente
    print(f"Usando GeoJSON actualizado: {ruta_entrada}")
elif ruta_shapefile.exists():
    ruta_entrada = ruta_shapefile
    print(f"Usando shapefile: {ruta_entrada}")
else:
    raise FileNotFoundError("No se encontro ningun archivo de entrada en Data/")

# Cargar y reproyectar a WGS84
gdf_corredor = gpd.read_file(ruta_entrada)
print(f"CRS original: {gdf_corredor.crs}")

if gdf_corredor.crs is None:
    gdf_corredor = gdf_corredor.set_crs(epsg=3115)

if gdf_corredor.crs.to_epsg() != 4326:
    gdf_corredor = gdf_corredor.to_crs(epsg=4326)

print(f"CRS final: {gdf_corredor.crs}")
print(f"Registros: {len(gdf_corredor)}")
gdf_corredor.head()

Usando GeoJSON actualizado: Corredor_San_fernando/Data/Geojson_Corredor_San_fernando/geojson_Corredor_San_fernando.geojson
CRS original: EPSG:3115
CRS final: EPSG:4326
Registros: 1


,Id,geometry
0,0,"MULTIPOLYGON (((-76.53468 3.42265, -76.53535 3..."


## 4. Validacion y exportacion a GeoJSON

In [4]:
# Validacion de geometrias
invalidas = gdf_corredor[~gdf_corredor.geometry.is_valid]
vacias = gdf_corredor[gdf_corredor.geometry.is_empty]

print("--- Reporte de calidad ---")
print(f"Registros: {len(gdf_corredor)}")
print(f"Geometrias invalidas: {len(invalidas)}")
print(f"Geometrias vacias: {len(vacias)}")
print(f"Bounds: {gdf_corredor.total_bounds}")

if len(invalidas) > 0:
    gdf_corredor['geometry'] = gdf_corredor.geometry.buffer(0)
    print("Geometrias corregidas con buffer(0).")

# Exportar a GeoJSON WGS84
gdf_corredor.to_file(ruta_geojson, driver='GeoJSON')
print(f"\nGeoJSON WGS84 exportado: {ruta_geojson}")

--- Reporte de calidad ---
Registros: 1
Geometrias invalidas: 0
Geometrias vacias: 0
Bounds: [-76.54983987   3.41250651 -76.53276244   3.43406679]

GeoJSON WGS84 exportado: Corredor_San_fernando/Data/Corredor_San_fernando_wgs84.geojson


## 5. Geovisor interactivo con edicion de poligono

El mapa carga el GeoJSON del corredor como capa **editable**.
Puedes mover los vertices del poligono para cambiar su forma y area.

**Instrucciones:**
1. Haz clic en el icono de edicion (lapiz) en la barra de herramientas izquierda
2. Los vertices del poligono se vuelven arrastrables -- muevalos a gusto
3. Haz clic en 'Save' cuando termines de editar
4. Haz clic en 'Export' (esquina superior derecha) para descargar el GeoJSON editado
5. Sube ese archivo en la celda siguiente para que el sistema lo reconozca

In [5]:
# Leer el GeoJSON exportado
with open(ruta_geojson) as f:
    geojson_data = json.load(f)

# Calcular centro del poligono
centroide = gdf_corredor.union_all().centroid
centro = [centroide.y, centroide.x]

# Crear mapa
mapa = folium.Map(location=centro, zoom_start=15, tiles='OpenStreetMap')

# Capas base adicionales
folium.TileLayer('CartoDB positron', name='CartoDB Claro').add_to(mapa)
folium.TileLayer('CartoDB dark_matter', name='CartoDB Oscuro').add_to(mapa)

# Crear FeatureGroup editable y agregar el poligono ahi
# Esto permite que Draw lo reconozca como editable (mover vertices)
fg_editable = folium.FeatureGroup(name='Poligono editable')
fg_editable.add_to(mapa)

# Herramienta de dibujo vinculada al FeatureGroup editable
draw = Draw(
    export=True,
    draw_options={
        'polyline': False,
        'rectangle': True,
        'polygon': True,
        'circle': False,
        'marker': False,
        'circlemarker': False
    },
    edit_options={'edit': True, 'remove': True}
)
draw.add_to(mapa)

# Inyectar JavaScript para cargar el GeoJSON en el FeatureGroup editable
# Esto hace que los vertices sean arrastrables con la herramienta de edicion
geojson_str = json.dumps(geojson_data)
js_code = f"""
<script>
    // Esperar a que el mapa se inicialice
    setTimeout(function() {{
        var mapElement = document.querySelector('.folium-map');
        var mapId = mapElement.id;
        var map = window[mapId] || null;

        // Buscar el FeatureGroup de drawn items (el primero que sea FeatureGroup)
        var drawnItems = null;
        map.eachLayer(function(layer) {{
            if (layer instanceof L.FeatureGroup && !(layer instanceof L.TileLayer)) {{
                if (drawnItems === null) {{
                    drawnItems = layer;
                }}
            }}
        }});

        // Si no lo encontramos por iteracion, buscar en variables globales
        if (!drawnItems) {{
            for (var key in window) {{
                if (key.startsWith('drawnItems_')) {{
                    drawnItems = window[key];
                    break;
                }}
            }}
        }}

        if (drawnItems) {{
            // Agregar el GeoJSON al FeatureGroup editable
            var geojson = {geojson_str};
            L.geoJSON(geojson).eachLayer(function(layer) {{
                drawnItems.addLayer(layer);
            }});
        }}
    }}, 1000);
</script>
"""

mapa.get_root().html.add_child(folium.Element(js_code))

folium.LayerControl(collapsed=False).add_to(mapa)

# Guardar HTML del mapa
ruta_mapa = ruta_outputs / 'geovisor_editable.html'
mapa.save(str(ruta_mapa))
print(f"Mapa guardado en: {ruta_mapa}")
print("Usa el icono de edicion (lapiz) para mover vertices.")
print("Luego Export para descargar el poligono modificado.")

mapa

Mapa guardado en: Corredor_San_fernando/Outputs/geovisor_editable.html
Usa el icono de edicion (lapiz) para mover vertices.
Luego Export para descargar el poligono modificado.


## 6. Cargar poligono editado

Si editaste el poligono en el mapa y descargaste el GeoJSON (boton Export),
subelo aqui para que el sistema lo reconozca como nueva area de estudio.

Si no editaste nada, se usara el poligono original.

In [ ]:
from google.colab import files
import io

print("Sube el archivo GeoJSON exportado del mapa (data.geojson).")
print("Si no editaste el poligono, simplemente ejecuta la siguiente celda.")
print()

try:
    uploaded = files.upload()
    if uploaded:
        nombre_archivo = list(uploaded.keys())[0]
        contenido = uploaded[nombre_archivo].decode('utf-8')
        geojson_editado = json.loads(contenido)
        gdf_area_estudio = gpd.GeoDataFrame.from_features(
            geojson_editado['features'], crs='EPSG:4326'
        )
        print(f"\nPoligono editado cargado: {len(gdf_area_estudio)} geometria(s)")
        # Guardar el poligono editado
        ruta_editado = ruta_base / 'Data' / 'poligono_editado.geojson'
        gdf_area_estudio.to_file(ruta_editado, driver='GeoJSON')
        print(f"Guardado en: {ruta_editado}")
    else:
        print("No se subio archivo. Se usara el poligono original.")
        gdf_area_estudio = gdf_corredor.copy()
except Exception as e:
    print(f"No se pudo cargar archivo: {e}")
    print("Se usara el poligono original.")
    gdf_area_estudio = gdf_corredor.copy()

## 6b. Usar poligono original (si no se edito)

Ejecuta esta celda solo si NO subiste un poligono editado arriba.

In [7]:
# Si no se definio gdf_area_estudio en la celda anterior, usar el original
if 'gdf_area_estudio' not in dir() or gdf_area_estudio is None:
    gdf_area_estudio = gdf_corredor.copy()

print(f"Area de estudio activa: {len(gdf_area_estudio)} poligono(s)")
print(f"Bounds: {gdf_area_estudio.total_bounds}")

Area de estudio activa: 1 poligono(s)
Bounds: [-76.54983987   3.41250651 -76.53276244   3.43406679]


## 7. Descarga del poligono actual

Descarga el poligono que se esta usando como area de estudio (original o editado).

In [ ]:
# Exportar area de estudio como GeoJSON descargable
ruta_descarga = ruta_outputs / 'area_estudio_actual.geojson'
gdf_area_estudio.to_file(ruta_descarga, driver='GeoJSON')

if EN_COLAB:
    files.download(str(ruta_descarga))
    print("Descarga iniciada.")
else:
    print(f"Archivo guardado en: {ruta_descarga}")

## 8. Extraccion de POIs de OpenStreetMap

Se usa la Overpass API para extraer puntos de interes (POIs) dentro del
area de estudio. Categorias: salud, educacion, comercio, transporte,
servicios publicos, recreacion.

In [11]:
def consultar_overpass(bbox, categorias):
    """
    Consulta la Overpass API para obtener POIs dentro de un bounding box.
    bbox: (sur, oeste, norte, este)
    categorias: dict con nombre_categoria: [(key, value), ...]
    """
    url = 'https://overpass-api.de/api/interpreter'
    resultados = []

    for categoria, tags in categorias.items():
        filtros = ''.join(
            f'node["{k}"="{v}"]({bbox[0]},{bbox[1]},{bbox[2]},{bbox[3]});'
            for k, v in tags
        )
        # Tambien buscar ways (edificios/areas)
        filtros_way = ''.join(
            f'way["{k}"="{v}"]({bbox[0]},{bbox[1]},{bbox[2]},{bbox[3]});'
            for k, v in tags
        )
        query = f"""
        [out:json][timeout:30];
        (
            {filtros}
            {filtros_way}
        );
        out center;
        """
        resp = requests.get(url, params={'data': query})
        if resp.status_code == 200:
            datos = resp.json()
            for elem in datos.get('elements', []):
                lat = elem.get('lat') or elem.get('center', {}).get('lat')
                lon = elem.get('lon') or elem.get('center', {}).get('lon')
                if lat and lon:
                    nombre = elem.get('tags', {}).get('name', 'Sin nombre')
                    subtipo = elem.get('tags', {}).get(
                        tags[0][0], ''
                    ) if tags else ''
                    resultados.append({
                        'nombre': nombre,
                        'categoria': categoria,
                        'subtipo': subtipo,
                        'latitud': lat,
                        'longitud': lon,
                        'osm_id': elem.get('id')
                    })
        else:
            print(f"Error en categoria '{categoria}': HTTP {resp.status_code}")

    return pd.DataFrame(resultados)


# Definir categorias de POIs
categorias_poi = {
    'Salud': [('amenity', 'hospital'), ('amenity', 'clinic'),
              ('amenity', 'pharmacy'), ('amenity', 'doctors')],
    'Educacion': [('amenity', 'school'), ('amenity', 'university'),
                  ('amenity', 'kindergarten'), ('amenity', 'college')],
    'Comercio': [('shop', 'supermarket'), ('shop', 'convenience'),
                 ('shop', 'mall'), ('amenity', 'marketplace')],
    'Transporte': [('amenity', 'bus_station'), ('highway', 'bus_stop'),
                   ('amenity', 'fuel'), ('amenity', 'parking')],
    'Recreacion': [('leisure', 'park'), ('leisure', 'sports_centre'),
                   ('leisure', 'playground'), ('amenity', 'community_centre')],
    'Servicios': [('amenity', 'bank'), ('amenity', 'police'),
                  ('amenity', 'fire_station'), ('amenity', 'post_office')]
}

# Obtener bounding box del area de estudio
bounds = gdf_area_estudio.total_bounds  # minx, miny, maxx, maxy
bbox_overpass = (float(bounds[1]), float(bounds[0]), float(bounds[3]), float(bounds[2]))  # sur, oeste, norte, este

print(f"Bounding box para consulta: {bbox_overpass}")
print("Consultando Overpass API...")

df_pois = consultar_overpass(bbox_overpass, categorias_poi)
print(f"\nTotal de POIs encontrados: {len(df_pois)}")

Bounding box para consulta: (np.float64(3.4125065078485197), np.float64(-76.54983987070719), np.float64(3.434066794519097), np.float64(-76.5327624447812))
Consultando Overpass API...
Error en categoria 'Salud': HTTP 406
Error en categoria 'Educacion': HTTP 406
Error en categoria 'Comercio': HTTP 406
Error en categoria 'Transporte': HTTP 406
Error en categoria 'Recreacion': HTTP 406
Error en categoria 'Servicios': HTTP 406

Total de POIs encontrados: 0


## 9. Filtrar POIs dentro del poligono

Los POIs del bounding box se filtran espacialmente para quedarse
solo con los que estan DENTRO del poligono del area de estudio.

In [12]:
if len(df_pois) > 0:
    # Crear GeoDataFrame de POIs
    gdf_pois = gpd.GeoDataFrame(
        df_pois,
        geometry=gpd.points_from_xy(df_pois.longitud, df_pois.latitud),
        crs='EPSG:4326'
    )

    # Filtrar solo los que estan dentro del poligono
    gdf_pois_dentro = gpd.sjoin(
        gdf_pois, gdf_area_estudio[['geometry']], predicate='within', how='inner'
    )
    # Limpiar columnas del join
    gdf_pois_dentro = gdf_pois_dentro.drop(columns=['index_right'], errors='ignore')

    print(f"POIs dentro del poligono: {len(gdf_pois_dentro)}")
    print(f"\nPOIs por categoria:")
    print(gdf_pois_dentro['categoria'].value_counts().to_string())
else:
    print("No se encontraron POIs en la zona.")
    gdf_pois_dentro = gpd.GeoDataFrame()

No se encontraron POIs en la zona.


## 10. Tabla de POIs en el area de estudio

Listado completo de puntos de interes encontrados dentro del poligono.

In [13]:
if len(gdf_pois_dentro) > 0:
    # Tabla formateada
    tabla = gdf_pois_dentro[[
        'nombre', 'categoria', 'subtipo', 'latitud', 'longitud'
    ]].sort_values(['categoria', 'nombre']).reset_index(drop=True)

    tabla.index = tabla.index + 1
    tabla.index.name = 'N'

    print(f"Total: {len(tabla)} puntos de interes\n")
    display(tabla)
else:
    print("No hay POIs para mostrar.")

No hay POIs para mostrar.


## 11. Mapa final con POIs clasificados por categoria

In [14]:
# Colores por categoria
colores_categoria = {
    'Salud': 'red',
    'Educacion': 'blue',
    'Comercio': 'green',
    'Transporte': 'orange',
    'Recreacion': 'purple',
    'Servicios': 'darkblue'
}

# Crear mapa final
mapa_final = folium.Map(location=centro, zoom_start=15, tiles='CartoDB positron')

# Agregar poligono del area de estudio
folium.GeoJson(
    gdf_area_estudio.to_json(),
    name='Area de estudio',
    style_function=lambda x: {
        'fillColor': '#3388ff',
        'color': '#2255cc',
        'weight': 2,
        'fillOpacity': 0.1
    }
).add_to(mapa_final)

# Agregar POIs por categoria como FeatureGroups
if len(gdf_pois_dentro) > 0:
    for categoria, color in colores_categoria.items():
        pois_cat = gdf_pois_dentro[gdf_pois_dentro['categoria'] == categoria]
        if len(pois_cat) == 0:
            continue
        fg = folium.FeatureGroup(name=f"{categoria} ({len(pois_cat)})")
        for _, poi in pois_cat.iterrows():
            folium.CircleMarker(
                location=[poi.latitud, poi.longitud],
                radius=6,
                color=color,
                fill=True,
                fill_color=color,
                fill_opacity=0.7,
                popup=f"<b>{poi.nombre}</b><br>{poi.categoria} - {poi.subtipo}",
                tooltip=poi.nombre
            ).add_to(fg)
        fg.add_to(mapa_final)

folium.LayerControl(collapsed=False).add_to(mapa_final)

mapa_final

## 12. Exportacion de resultados

Se exportan los POIs como GeoJSON y CSV, y el mapa final como HTML.

In [ ]:
if len(gdf_pois_dentro) > 0:
    # Exportar POIs como GeoJSON
    ruta_pois_geojson = ruta_outputs / 'pois_area_estudio.geojson'
    gdf_pois_dentro.to_file(ruta_pois_geojson, driver='GeoJSON')
    print(f"POIs GeoJSON: {ruta_pois_geojson}")

    # Exportar como CSV
    ruta_pois_csv = ruta_outputs / 'pois_area_estudio.csv'
    gdf_pois_dentro.drop(columns='geometry').to_csv(ruta_pois_csv, index=False)
    print(f"POIs CSV: {ruta_pois_csv}")

# Exportar mapa HTML final
ruta_mapa_final = ruta_outputs / 'geovisor_pois_san_fernando.html'
mapa_final.save(str(ruta_mapa_final))
print(f"Mapa HTML: {ruta_mapa_final}")

# Descargar archivos en Colab
if EN_COLAB and len(gdf_pois_dentro) > 0:
    print("\nDescargando archivos...")
    files.download(str(ruta_pois_csv))
    files.download(str(ruta_mapa_final))